In [33]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt,Command
from typing import Annotated, TypedDict
from dotenv import load_dotenv
import os

load_dotenv()


True

In [34]:
token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct",
    provider="featherless-ai",
    task="text-generation",
    huggingfacehub_api_token=token,
)
llm = ChatHuggingFace(llm=endpoint)


In [35]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def chat_node(state: ChatState):
    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })

    approval = decision.get("approved", "") if isinstance(decision, dict) else decision
    if isinstance(approval, str) and approval.strip().lower() in {"y", "yes"}:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    return {"messages": [AIMessage(content="Question not approved. Please ask another question.")]}


In [36]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)
checkpointer = MemorySaver()
bot = graph.compile(checkpointer=checkpointer)


In [37]:
config = {"configurable": {"thread_id": "1234"}}
initial_input = {
    "messages": [HumanMessage(content="Explain gradient descent in very simple terms.")]
}

# This pauses the graph and will raise a GraphInterrupt until resumed
result = bot.invoke(initial_input, config=config)


In [38]:
result

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='bc386ddf-b41c-4d29-9bbe-1de9af61a1cf')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question? yes/no'}, id='f564c50a8b3b12876c76d92628b0ea14')]}

In [39]:
message = result["__interrupt__"][0].value

In [40]:
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'Explain gradient descent in very simple terms.',
 'instruction': 'Approve this question? yes/no'}

In [41]:
user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")

In [ ]:
final_result = bot.invoke(
    Command(resume={"approved": user_input.strip().lower()}),
    config=config,
)


In [ ]:
result = final_result["messages"][-1].content

In [ ]:
result